In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.listdir('/content/drive/MyDrive/Colab')

['placement_dataset.csv',
 'placement_predict_50k Dataset (1).csv',
 'placement_predict_50k_adjusted.csv',
 'age_insurance.csv',
 'weather_cricket_data.csv']

In [8]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42

# ---------------------------------------------------------------------
# 1. Load the simple dataset
# ---------------------------------------------------------------------
df = pd.read_csv("/content/drive/MyDrive/Colab/placement_dataset.csv")

print("Sample of the dataset:")
print(df.head())
print(f"\nDataset shape: {df.shape}")
print(f"Class balance:\n{df['Placed'].value_counts()}")

TARGET = "Placed"
DROP_COLS = ["StudentID"] # just an id, not a predictor

# Separate features and target
feature_cols = [c for c in df.columns if c not in [TARGET] + DROP_COLS]
X = df[feature_cols].copy() # Make a copy to avoid SettingWithCopyWarning
y = df[TARGET]

# Identify categorical and numerical features
categorical_features = X.select_dtypes(include='object').columns
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns

# Handle missing numerical values with mean imputation
imputer = SimpleImputer(strategy='mean')
X.loc[:, numerical_features] = imputer.fit_transform(X[numerical_features]) # Use .loc to avoid SettingWithCopyWarning

# Apply one-hot encoding to categorical features
X = pd.get_dummies(X, columns=categorical_features, drop_first=True)

# Update feature names after encoding
feature_names = X.columns.tolist()

# Convert X to numpy array for sklearn compatibility
X = X.values
y = y.values # Ensure y is also a numpy array

# ---------------------------------------------------------------------
# 2. Train / test split
# ---------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=RANDOM_STATE
)

# ---------------------------------------------------------------------
# 3. Train a Random Forest
# ---------------------------------------------------------------------
rf = RandomForestClassifier(
n_estimators=100,
max_features="sqrt",
oob_score=True,
random_state=RANDOM_STATE,
)
rf.fit(X_train, y_train)

# ---------------------------------------------------------------------
# 4. Evaluate
# ---------------------------------------------------------------------
y_pred = rf.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)

print(f"\nOOB score: {rf.oob_score_:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred))

# ---------------------------------------------------------------------
# 5. Feature importances
# ---------------------------------------------------------------------
print("Feature importances:")
for name, importance in sorted(
zip(feature_names, rf.feature_importances_), key=lambda x: -x[1]
):
    print(f" {name}: {importance:.4f}")

Sample of the dataset:
   CGPA     IQ  Aptitude_Score  Communication_Skill  Internships Branch Placed
0  8.00   81.0            89.0                    1            1   MECH     No
1  7.36  105.0            74.0                    5            3  CIVIL    Yes
2  8.15   96.0            62.0                    6            1     IT     No
3  9.02  119.0            56.0                    6            2    CSE     No
4   NaN    NaN             NaN                    3            3    CSE    Yes

Dataset shape: (200, 7)
Class balance:
Placed
Yes    117
No      83
Name: count, dtype: int64

OOB score: 0.5188
Test accuracy: 0.5000

Classification report:
              precision    recall  f1-score   support

          No       0.40      0.35      0.38        17
         Yes       0.56      0.61      0.58        23

    accuracy                           0.50        40
   macro avg       0.48      0.48      0.48        40
weighted avg       0.49      0.50      0.49        40

Feature importan

In [9]:
rf = RandomForestClassifier(
 n_estimators=100,
max_features="sqrt",
oob_score=True,
random_state=RANDOM_STATE,
)